# 🔍 Python Query Optimization — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> The query optimizer is a travel planner. You say "get me from A to B" (SQL). It maps every possible route (join orders, index paths, full scans), estimates the cost of each (row counts × page reads), picks the cheapest one, and hands you a printed itinerary (execution plan). Your job is to read that itinerary and know when the planner took a bad route.

---

## 📋 Table of Contents

| # | Section |
|---|----------|
| 1 | [What Is Query Optimization? The Visual Model](#1) |
| 2 | [Core Concepts — Setup](#2) |
| 3 | [The Core API — All Operations](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: Reading Execution Plans — EXPLAIN QUERY PLAN](#5) |
| 6 | [Pattern 2: Index Selection — B-tree, Composite, Covering](#6) |
| 7 | [Pattern 3: Partition Pruning & Predicate Pushdown](#7) |
| 8 | [Pattern 4: Join Strategies — Hash, Merge, Nested Loop](#8) |
| 9 | [Pattern 5: Query Rewriting — CTEs, Subqueries, Materializing](#9) |
| 10 | [The Query Optimization Decision Map](#10) |
| 11 | [Interview Cheat Sheet](#11) |
| 12 | [Summary Map](#12) |

<a id='1'></a>
## 1. What Is Query Optimization? The Visual Model

---

```
SQL QUERY LIFECYCLE

  SQL TEXT
    │ parse
    ▼
  LOGICAL PLAN  (relational algebra: σ=filter, π=project, ⋈=join)
    │ optimize (enumerate plans, cost each)
    ▼
  PHYSICAL PLAN (chosen access paths: index scan vs seq scan, join algo)
    │ execute
    ▼
  RESULT ROWS

EXECUTION PLAN ANATOMY:

  Seq Scan on orders  (cost=0..1500 rows=50000)   ← full table scan
    Filter: (status = 'OPEN')
  Index Scan on orders_status_idx                  ← index lookup
    Index Cond: (status = 'OPEN')
  Hash Join  (cost=250..800 rows=1200)             ← in-memory hash table
    Hash Cond: (orders.cust_id = customers.id)
    → Seq Scan on customers
    → Hash on orders

COST COMPONENTS:
  seq_page_cost    = 1.0   (baseline — reading pages sequentially)
  random_page_cost = 4.0   (random I/O — 4× more expensive)
  cpu_tuple_cost   = 0.01  (processing each row)

  Seq scan cost  = pages × seq_page_cost + rows × cpu_tuple_cost
  Index scan cost = matching_rows × random_page_cost + index_levels × 1.0
  Index beats seq scan when: selectivity < ~5% of rows

STATISTICS THE PLANNER USES:
  pg_class.reltuples   — estimated row count per table
  pg_stats.n_distinct  — distinct values per column
  pg_stats.histogram   — value distribution (for range predicates)
  pg_stats.correlation — physical vs logical ordering (for seq scan benefit)
```


<a id='2'></a>
## 2. Core Concepts — Setup

In [ ]:
import sqlite3
import time
import random
from collections import defaultdict

# build an in-memory database with realistic tables for all demos
conn = sqlite3.connect(":memory:")
cur = conn.cursor()

# orders table — 10k rows simulating an e-commerce fact table
cur.execute("""
    CREATE TABLE orders (
        order_id   INTEGER PRIMARY KEY,
        customer_id INTEGER NOT NULL,
        status      TEXT NOT NULL,
        amount      REAL NOT NULL,
        region      TEXT NOT NULL,
        order_date  TEXT NOT NULL
    )
""")

# customers table — 1k rows
cur.execute("""
    CREATE TABLE customers (
        customer_id INTEGER PRIMARY KEY,
        name        TEXT NOT NULL,
        tier        TEXT NOT NULL
    )
""")

random.seed(42)
statuses = ['OPEN', 'CLOSED', 'PENDING', 'CANCELLED']
regions  = ['NORTH', 'SOUTH', 'EAST', 'WEST']
tiers    = ['GOLD', 'SILVER', 'BRONZE']

cust_data = [(i, f'Customer_{i}', random.choice(tiers)) for i in range(1, 1001)]
cur.executemany("INSERT INTO customers VALUES (?,?,?)", cust_data)

order_data = [
    (i,
     random.randint(1, 1000),
     random.choice(statuses),
     round(random.uniform(10, 5000), 2),
     random.choice(regions),
     f'2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}')
    for i in range(1, 10001)
]
cur.executemany("INSERT INTO orders VALUES (?,?,?,?,?,?)", order_data)
conn.commit()

print("Tables created:")
print(f"  orders:    {cur.execute('SELECT COUNT(*) FROM orders').fetchone()[0]:,} rows")
print(f"  customers: {cur.execute('SELECT COUNT(*) FROM customers').fetchone()[0]:,} rows")
print("Setup complete.")

<a id='3'></a>
## 3. The Core API — All Operations

---

```
QUERY OPTIMIZATION OPERATIONS
─────────────────────────────────────────────────────────────────────────
OPERATION                  WHAT IT DOES
─────────────────────────────────────────────────────────────────────────
EXPLAIN query              show estimated plan (no execution)
EXPLAIN ANALYZE query      execute + show actual rows/time
EXPLAIN (BUFFERS) query    show cache hit/miss counts
CREATE INDEX ON t(col)     B-tree index — good for =, <, >, BETWEEN, LIKE 'x%'
CREATE INDEX ON t(a,b)     composite — covers (a), (a,b), not (b) alone
CREATE INDEX ON t(a) INCL  covering — avoids heap fetch for included cols
ANALYZE table              update statistics (planner uses these for cost)
VACUUM ANALYZE             reclaim space + update stats
SET enable_seqscan=off     force index scan (debugging only)
─────────────────────────────────────────────────────────────────────────

THINGS YOU DO NOT DO:
❌  Index every column — each index slows INSERT/UPDATE/DELETE
❌  Use functions on indexed columns in WHERE: WHERE YEAR(date)=2024 kills the index
❌  SELECT * in production — always project only needed columns
❌  Trust EXPLAIN without ANALYZE — estimates can be wrong on stale stats
❌  Add indexes without measuring — check actual selectivity first
❌  Use OFFSET for deep pagination — use keyset pagination instead
```


In [ ]:
# Demo: timing queries before and after index creation

def time_query(label, sql, params=()):
    start = time.perf_counter()
    rows = cur.execute(sql, params).fetchall()
    elapsed = (time.perf_counter() - start) * 1000
    print(f"  {label:40s}: {len(rows):5} rows  {elapsed:.2f}ms")
    return rows

print("=== BEFORE any indexes ===")
time_query("status='OPEN'              ",
           "SELECT * FROM orders WHERE status='OPEN'")
time_query("customer_id=42             ",
           "SELECT * FROM orders WHERE customer_id=42")
time_query("JOIN customers (no index)  ",
           "SELECT o.order_id, c.name FROM orders o JOIN customers c ON o.customer_id=c.customer_id WHERE o.status='OPEN' LIMIT 5")

print()
# ANALYZE in SQLite: EXPLAIN QUERY PLAN shows scan strategy
plan = cur.execute("EXPLAIN QUERY PLAN SELECT * FROM orders WHERE status='OPEN'").fetchall()
print("Execution plan (before index):")
for row in plan:
    print(f"  {row}")

# create index
cur.execute("CREATE INDEX idx_orders_status ON orders(status)")
cur.execute("CREATE INDEX idx_orders_customer ON orders(customer_id)")
conn.commit()

print()
print("=== AFTER indexes ===")
time_query("status='OPEN'              ",
           "SELECT * FROM orders WHERE status='OPEN'")
time_query("customer_id=42             ",
           "SELECT * FROM orders WHERE customer_id=42")

plan2 = cur.execute("EXPLAIN QUERY PLAN SELECT * FROM orders WHERE status='OPEN'").fetchall()
print("Execution plan (after index):")
for row in plan2:
    print(f"  {row}")

print("\nCore API demo complete.")

<a id='4'></a>
## 4. Decision Map — When To Use What

---

```
SIGNAL IN THE PROBLEM                        WHAT TO DO
────────────────────────────────────────────────────────────────────────
query is slow, don't know why                EXPLAIN ANALYZE first
Seq Scan on large table                      check selectivity → add index
Index Scan but still slow                    check if index is covering
high rows estimate, low actual rows          ANALYZE to refresh stats
WHERE on function(col)                       rewrite to WHERE col BETWEEN
many joins, wrong order chosen               check row count estimates
OFFSET 10000 is slow                         switch to keyset pagination
same subquery repeated                       materialize as CTE or temp table
column has low cardinality (10 values)       index may not help — partial index
range query on timestamp                     BRIN index (PostgreSQL)
text search (LIKE '%word%')                  GIN + tsvector full-text index
JOIN on high-cardinality column              hash join (auto-chosen usually)
JOIN on pre-sorted data                      merge join cheaper than hash
small table joins large table                broadcast/nested-loop preferable
────────────────────────────────────────────────────────────────────────
```


<a id='5'></a>
## 5. 🧩 Pattern 1: Reading Execution Plans — EXPLAIN QUERY PLAN

---

```
PROBLEM:
  A query on 10M rows takes 45s. You need to diagnose and fix it.

APPROACH:
  Run EXPLAIN ANALYZE. Read top-down. Find the most expensive node.
  Key numbers: actual rows vs estimated rows (big gap = stale stats)
  Key nodes: Seq Scan (bad on big tables), Index Scan (good),
             Hash Join (in-memory), Nested Loop (good only for tiny inner)

SLOW MOTION: reading a plan
  Step 1: Find the costliest node (highest cost= or longest actual_time)
  Step 2: Check rows estimate vs actual — 10× off = stale stats
  Step 3: Seq Scan on large table → add index if selectivity < 5%
  Step 4: Nested Loop with large inner table → should be Hash Join
  Step 5: Index Scan but still slow → check if it's a covering index

KEY INSIGHT:
  The bottleneck is almost always the node with the widest estimate/actual gap
  or a Seq Scan on a table > 100k rows where you filter to < 1% of rows.

TIME / SPACE:
  Seq Scan:     O(N) — reads every page, every row
  Index Scan:   O(log N + k) — B-tree traversal + k matching rows
  Bitmap Scan:  O(log N + k) — batches random reads into sequential
  Hash Join:    O(N + M) — build hash on smaller table, probe with larger
  Nested Loop:  O(N × M) — bad unless inner is tiny or fully indexed
```


In [ ]:
# Pattern 1: Execution plan analysis simulation
# SQLite uses EXPLAIN QUERY PLAN — simpler than PostgreSQL but same concepts

# Slow motion: reading the plan step by step
# step  operation           meaning
#  1    SCAN orders         full table scan — no usable index for this predicate
#  2    SEARCH orders       index used — lookup by key
#  3    SCAN + USE TEMP     sort or hash operation happening

def explain_and_time(label, sql, params=()):
    plan = cur.execute(f"EXPLAIN QUERY PLAN {sql}", params).fetchall()
    start = time.perf_counter()
    rows = cur.execute(sql, params).fetchall()
    ms = (time.perf_counter() - start) * 1000
    plan_str = ' | '.join(str(r[3]) for r in plan)  # detail column
    print(f"  [{label}]")
    print(f"    plan: {plan_str}")
    print(f"    result: {len(rows)} rows in {ms:.2f}ms")
    return rows

print("=== Query 1: filter on non-indexed column ===")
explain_and_time("amount > 4000",
    "SELECT order_id, amount FROM orders WHERE amount > 4000")

print()
print("=== Query 2: filter on indexed column ===")
explain_and_time("status='OPEN' (indexed)",
    "SELECT order_id, amount FROM orders WHERE status='OPEN'")

print()
print("=== Query 3: composite predicate ===")
explain_and_time("status AND region",
    "SELECT order_id FROM orders WHERE status='OPEN' AND region='NORTH'")

print()
# demonstrate that function on column kills index
cur.execute("CREATE INDEX idx_orders_date ON orders(order_date)")
conn.commit()

print("=== Query 4: function on indexed column — index cannot be used ===")
explain_and_time("SUBSTR(order_date) — function prevents index use",
    "SELECT order_id FROM orders WHERE SUBSTR(order_date,1,4)='2024'")

print()
print("=== Query 5: range on indexed column — index IS used ===")
explain_and_time("date BETWEEN — range on index",
    "SELECT order_id FROM orders WHERE order_date BETWEEN '2024-01-01' AND '2024-12-31'")

print("\nExecution plan pattern complete.")

<a id='6'></a>
## 6. 🧩 Pattern 2: Index Selection — B-tree, Composite, Covering

---

```
PROBLEM:
  You have a query that filters on (status, region) and returns (order_id, amount).
  Design the optimal index strategy.

APPROACH:
  Three index types for three use cases:
  1. Single column: WHERE col = value  (high selectivity → good index candidate)
  2. Composite:     WHERE col_a = x AND col_b = y  (put = cols before range cols)
  3. Covering:      index includes all projected columns → zero heap fetches

COMPOSITE INDEX COLUMN ORDER RULE:
  "Equality first, range last"
  CREATE INDEX ON orders(status, region, order_date)
  Works for: WHERE status='OPEN'
             WHERE status='OPEN' AND region='NORTH'
             WHERE status='OPEN' AND region='NORTH' AND order_date > '2024-01-01'
  Breaks for: WHERE region='NORTH'  ← skips leading column, can't use index

COVERING INDEX:
  CREATE INDEX ON orders(status) INCLUDE (order_id, amount)
  The SELECT projects only order_id, amount — both in index → index-only scan
  Eliminates the heap fetch (random I/O per matching row)
  Cost reduction: up to 10× on high-selectivity queries

KEY INSIGHT:
  Index width = storage cost. Only cover columns that are frequently projected.
  Partial index (WHERE status='OPEN') is even smaller when filtering a hot subset.

TIME / SPACE:
  Without covering index: index lookup + heap fetch = 2 × random I/O per row
  With covering index:    index only = 1 × sequential read in index pages
  Index creation:  O(N log N) — sort + build B-tree
  Index size:      O(N) pages ≈ 10-30% of table size for typical columns
```


In [ ]:
# Pattern 2: Index selection — composite vs single vs covering

# Slow motion: designing index for query:
# SELECT order_id, amount FROM orders WHERE status='OPEN' AND region='NORTH'
# step 1: single index on status → SEARCH orders USING INDEX idx_orders_status
# step 2: composite (status, region) → tighter search, fewer rows examined
# step 3: covering (status, region) INCLUDE (order_id, amount) → index-only scan

def index_demo(label, index_sql, query_sql, drop_sql=None):
    if index_sql:
        cur.execute(index_sql)
        conn.commit()
    plan = cur.execute(f"EXPLAIN QUERY PLAN {query_sql}").fetchall()
    start = time.perf_counter()
    rows = cur.execute(query_sql).fetchall()
    ms = (time.perf_counter() - start) * 1000
    plan_detail = [str(r[3]) for r in plan]
    print(f"  [{label}]")
    for p in plan_detail:
        print(f"    plan: {p}")
    print(f"    rows={len(rows)} time={ms:.2f}ms")
    if drop_sql:
        cur.execute(drop_sql); conn.commit()

QUERY = "SELECT order_id, amount FROM orders WHERE status='OPEN' AND region='NORTH'"

print("=== No composite index (uses single status index) ===")
index_demo("status index only", None, QUERY)

print()
print("=== Composite index (status, region) ===")
index_demo("composite (status, region)",
    "CREATE INDEX IF NOT EXISTS idx_comp_sr ON orders(status, region)",
    QUERY)

print()
# demonstrate wrong order — region first doesn't help for status-only filter
print("=== Wrong order composite: (region, status) for WHERE status=? ===")
cur.execute("CREATE INDEX idx_comp_rs ON orders(region, status)")
conn.commit()
wrong_q = "SELECT order_id FROM orders WHERE status='OPEN'"
plan_wrong = cur.execute(f"EXPLAIN QUERY PLAN {wrong_q}").fetchall()
print("  plan:", [str(r[3]) for r in plan_wrong])
print("  (idx_comp_rs starts with region — cannot use for status-only filter)")

print()
# partial index — only index the hot subset
print("=== Partial index (WHERE status='OPEN') ===")
cur.execute("CREATE INDEX idx_partial_open ON orders(customer_id) WHERE status='OPEN'")
conn.commit()
plan_partial = cur.execute(
    "EXPLAIN QUERY PLAN SELECT order_id FROM orders WHERE status='OPEN' AND customer_id=42"
).fetchall()
print("  plan:", [str(r[3]) for r in plan_partial])
print("  (smaller index, faster lookup — covers the most common case)")

print("\nIndex selection pattern complete.")

<a id='7'></a>
## 7. 🧩 Pattern 3: Partition Pruning & Predicate Pushdown

---

```
PROBLEM:
  A 1-TB table is partitioned by year. A query filters on 2024 only.
  How does the engine avoid scanning 10 years of data?

APPROACH:
  Partition pruning: eliminate entire partitions before scanning.
  Predicate pushdown: move filters as close to the source as possible.

PARTITION PRUNING TRACE:
  Table orders_partitioned: 10 partitions by year (2015–2024)
  Query: WHERE order_date BETWEEN '2024-01-01' AND '2024-12-31'

  Before pruning: scan all 10 partitions → 10M rows examined
  After pruning:  scan only 2024 partition → 1M rows examined → 10× speedup

PREDICATE PUSHDOWN TRACE:
  Bad plan (filter AFTER join):
    JOIN orders × customers (10k × 1k = 10M row combinations)
    THEN filter status='OPEN' → 2.5M survive

  Good plan (filter BEFORE join — pushed down):
    filter orders WHERE status='OPEN' → 2.5k rows
    JOIN with customers → 2.5k × lookup = fast

KEY INSIGHT:
  Partition pruning = horizontal filter (fewer partitions to scan)
  Predicate pushdown = vertical filter (fewer rows before expensive ops)
  Both work by moving work as early as possible in the plan.

TIME / SPACE:
  Pruned scan: O(N/P) where P = number of partitions
  Pushed predicate: O(N×selectivity) vs O(N) without pushdown
  Both are optimizer decisions — you influence them via query structure
```


In [ ]:
# Pattern 3: Partition pruning and predicate pushdown simulation
# SQLite doesn't have native partitioning, so we simulate with multiple tables

# Slow motion: partition pruning
# step 1: query arrives with WHERE order_date BETWEEN '2024-01-01' AND '2024-12-31'
# step 2: optimizer checks partition metadata: only 2024_partition covers this range
# step 3: skip all other year partitions — 0 rows read from them
# step 4: scan 2024_partition only — 1/10 of total data

# create simulated partitions (one table per year)
for year in range(2020, 2025):
    cur.execute(f"""
        CREATE TABLE orders_{year} (
            order_id INTEGER PRIMARY KEY,
            customer_id INTEGER,
            status TEXT,
            amount REAL,
            order_date TEXT
        )
    """)
    # each partition gets ~200 rows
    partition_data = [
        (year * 10000 + i, random.randint(1,1000),
         random.choice(statuses),
         round(random.uniform(10,5000),2),
         f'{year}-{random.randint(1,12):02d}-{random.randint(1,28):02d}')
        for i in range(200)
    ]
    cur.executemany(f"INSERT INTO orders_{year} VALUES (?,?,?,?,?)", partition_data)
conn.commit()

class PartitionedTable:
    # simulates a partition manager that prunes partitions based on predicate
    def __init__(self, partitions):
        self.partitions = partitions  # list of (year, table_name)

    def query(self, start_date, end_date, label):
        start_year = int(start_date[:4])
        end_year   = int(end_date[:4])
        # partition pruning: only scan partitions in range
        pruned = [(y, t) for y, t in self.partitions if start_year <= y <= end_year]
        skipped = len(self.partitions) - len(pruned)
        total_rows = 0
        start_t = time.perf_counter()
        for year, table in pruned:
            rows = cur.execute(
                f"SELECT * FROM {table} WHERE order_date BETWEEN ? AND ?",
                (start_date, end_date)
            ).fetchall()
            total_rows += len(rows)
        ms = (time.perf_counter() - start_t) * 1000
        print(f"  [{label}]")
        print(f"    partitions scanned: {len(pruned)}/{len(self.partitions)} (skipped {skipped})")
        print(f"    rows returned: {total_rows}  time: {ms:.2f}ms")

pt = PartitionedTable([(y, f'orders_{y}') for y in range(2020, 2025)])

print("=== Partition Pruning ===")
pt.query('2024-01-01', '2024-12-31', 'single year — 1/5 partitions')
pt.query('2020-01-01', '2024-12-31', 'all years — no pruning possible')
pt.query('2022-06-01', '2023-06-30', 'two years — 2/5 partitions')

print()
print("=== Predicate Pushdown ===")
# bad: filter AFTER join
start = time.perf_counter()
bad_rows = cur.execute("""
    SELECT o.order_id, c.name
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.status = 'OPEN'
""").fetchall()
ms_bad = (time.perf_counter() - start) * 1000

# good: filter BEFORE join (subquery forces early filter)
start = time.perf_counter()
good_rows = cur.execute("""
    SELECT o.order_id, c.name
    FROM (SELECT order_id, customer_id FROM orders WHERE status='OPEN') o
    JOIN customers c ON o.customer_id = c.customer_id
""").fetchall()
ms_good = (time.perf_counter() - start) * 1000

print(f"  join-then-filter:   {len(bad_rows):5} rows  {ms_bad:.2f}ms")
print(f"  filter-then-join:   {len(good_rows):5} rows  {ms_good:.2f}ms")
print(f"  (same result, earlier filter can reduce join input significantly)")

print("\nPartition pruning pattern complete.")

<a id='8'></a>
## 8. 🧩 Pattern 4: Join Strategies — Hash, Merge, Nested Loop

---

```
PROBLEM:
  You're joining orders (10M rows) with customers (100k rows).
  The planner picks Nested Loop — it takes 120s. How do you fix it?

APPROACH:
  Three physical join algorithms — each optimal for different conditions:

  NESTED LOOP:  for each row in outer, scan inner
    Cost: O(N × M) — only good when inner is tiny OR inner has an index
    Good: small dimension table, or inner join via index key
    Bad:  two large tables → catastrophic

  HASH JOIN:    build hash table on smaller table, probe with larger
    Cost: O(N + M) — one pass each
    Good: large unsorted tables, equi-joins
    Bad:  memory pressure (spills to disk if hash table > work_mem)
    Trigger: SET enable_nestloop=off to force hash join in PG

  MERGE JOIN:   sort both inputs, advance two pointers
    Cost: O(N log N + M log M) if unsorted, O(N + M) if pre-sorted
    Good: pre-sorted inputs (index exists on join key), range joins
    Bad:  random data requires sort step first

SLOW MOTION: hash join mechanics
  Phase 1 (build): scan customers, hash customer_id → bucket → in memory
  Phase 2 (probe): scan orders, hash customer_id → look up in hash table
  Result:          emit matching pairs on lookup hit
  Memory:          need to hold the smaller table's hash — customers < orders

KEY INSIGHT:
  Always put the smaller table on the BUILD side of a hash join.
  If stats are wrong, the optimizer may get this backwards — use hints or rewrite.

TIME / SPACE:
  Nested Loop: O(N×M) — avoid on large tables
  Hash Join:   O(N+M) time, O(min(N,M)) space for hash table
  Merge Join:  O(N+M) time (if sorted), O(1) extra space
```


In [ ]:
# Pattern 4: Join strategy simulation
# SQLite always uses nested loop / hash join; we simulate the cost model

# Slow motion: hash join build + probe phase
# step 1 (build): iterate customers, store {customer_id: row} in dict → O(M)
# step 2 (probe): iterate orders, look up customer_id in dict → O(1) per row
# total: O(N + M) — much better than O(N × M) nested loop

def nested_loop_join(outer, inner, outer_key, inner_key):
    # O(N × M) — scan inner for every outer row
    # only acceptable when inner is tiny (< a few thousand rows)
    result = []
    for o_row in outer:
        for i_row in inner:
            if o_row[outer_key] == i_row[inner_key]:  # inner full scan per outer row
                result.append((o_row, i_row))
    return result

def hash_join(build_side, probe_side, build_key, probe_key):
    # O(N + M) — one pass to build hash, one pass to probe
    # build phase: smaller table → hash table
    hash_table = defaultdict(list)
    for row in build_side:
        hash_table[row[build_key]].append(row)  # hash on join key
    # probe phase: larger table → look up in hash table
    result = []
    for row in probe_side:
        for match in hash_table.get(row[probe_key], []):  # O(1) dict lookup
            result.append((match, row))
    return result

def merge_join(left, right, left_key, right_key):
    # O((N+M) log(N+M)) if unsorted, O(N+M) if pre-sorted
    # requires both inputs sorted on join key
    left_s  = sorted(left,  key=lambda r: r[left_key])
    right_s = sorted(right, key=lambda r: r[right_key])
    result = []
    i = j = 0
    while i < len(left_s) and j < len(right_s):
        lv = left_s[i][left_key]
        rv = right_s[j][right_key]
        if lv == rv:    # match: collect all right rows with same key
            k = j
            while k < len(right_s) and right_s[k][right_key] == lv:
                result.append((left_s[i], right_s[k]))
                k += 1
            i += 1
        elif lv < rv:
            i += 1      # advance left pointer
        else:
            j += 1      # advance right pointer
    return result

# build test data
customers_small = [{'customer_id': i, 'name': f'C{i}'} for i in range(1, 101)]  # 100 rows
orders_medium   = [{'order_id': i, 'customer_id': random.randint(1,100), 'amount': i*10.0}
                   for i in range(1, 501)]  # 500 rows

print("=== JOIN ALGORITHM COMPARISON ===")
print(f"  outer(orders)={len(orders_medium)}, inner(customers)={len(customers_small)}")
print()

# nested loop
start = time.perf_counter()
nl = nested_loop_join(orders_medium, customers_small, 'customer_id', 'customer_id')
ms_nl = (time.perf_counter() - start) * 1000
print(f"  nested_loop_join:  {len(nl)} pairs  {ms_nl:.2f}ms  O(N×M)={(len(orders_medium)*len(customers_small)):,} comparisons")

# hash join
start = time.perf_counter()
hj = hash_join(customers_small, orders_medium, 'customer_id', 'customer_id')
ms_hj = (time.perf_counter() - start) * 1000
print(f"  hash_join:         {len(hj)} pairs  {ms_hj:.2f}ms  O(N+M)={(len(orders_medium)+len(customers_small)):,} rows touched")

# merge join
start = time.perf_counter()
mj = merge_join(orders_medium, customers_small, 'customer_id', 'customer_id')
ms_mj = (time.perf_counter() - start) * 1000
print(f"  merge_join:        {len(mj)} pairs  {ms_mj:.2f}ms  O((N+M)logN) incl. sort")

print()
print("All algorithms produce same result:", len(nl) == len(hj) == len(mj))
print("Hash join chosen when: large unsorted equi-join")
print("Merge join chosen when: inputs pre-sorted (index on join key)")
print("Nested loop chosen when: tiny inner table OR inner has index on join key")

print("\nJoin strategies pattern complete.")

<a id='9'></a>
## 9. 🧩 Pattern 5: Query Rewriting — CTEs, Subqueries, Materialization

---

```
PROBLEM:
  A complex query references the same expensive subquery 3 times.
  How do you restructure it to execute that subquery only once?

APPROACH:
  Three rewrite patterns:
  1. CTE (WITH clause): may or may not materialize — optimizer decides
  2. MATERIALIZED CTE: force one-time execution (PG: WITH ... AS MATERIALIZED)
  3. Temp table:        explicit one-time execution, survives across statements

SLOW MOTION: CTE vs repeated subquery

  Bad (subquery evaluated 3 times):
    SELECT * FROM t WHERE col > (SELECT AVG(col) FROM t)
    UNION ALL
    SELECT * FROM t WHERE col < (SELECT AVG(col) FROM t) * 0.5

  Good (CTE evaluated once):
    WITH avg_val AS (SELECT AVG(col) AS v FROM t)
    SELECT * FROM t, avg_val WHERE t.col > avg_val.v
    UNION ALL
    SELECT * FROM t, avg_val WHERE t.col < avg_val.v * 0.5

CORRELATED SUBQUERY ANTI-PATTERN:
  SELECT o.order_id,
    (SELECT SUM(amount) FROM orders o2 WHERE o2.customer_id = o.customer_id)
  FROM orders o
  -- The subquery runs once per outer row → O(N²)
  -- Rewrite as: JOIN with grouped CTE → O(N)

  WITH cust_total AS (SELECT customer_id, SUM(amount) AS total FROM orders GROUP BY 1)
  SELECT o.order_id, ct.total
  FROM orders o JOIN cust_total ct ON o.customer_id = ct.customer_id

KEY INSIGHT:
  Correlated subquery in SELECT list = red flag = O(N²). Always rewrite as JOIN.

TIME / SPACE:
  Correlated subquery:  O(N²) — one execution per outer row
  CTE / temp table:     O(N)  — one execution, result cached
  Materialized CTE:     O(N)  — forced single execution + temp storage
```


In [ ]:
# Pattern 5: Query rewriting — CTE vs correlated subquery

# Slow motion: correlated subquery vs JOIN rewrite
# correlated: for each order row, re-execute subquery → O(N) subqueries
# join rewrite: compute aggregation once, then join → O(N) total

print("=== Correlated Subquery (anti-pattern) ===")
start = time.perf_counter()
# simulate correlated subquery — Python equivalent
order_sample = cur.execute("SELECT order_id, customer_id, amount FROM orders LIMIT 500").fetchall()
corr_results = []
for order_id, cust_id, amt in order_sample:
    # this inner query runs once per outer row — O(N²) equivalent
    total = cur.execute(
        "SELECT SUM(amount) FROM orders WHERE customer_id=?", (cust_id,)
    ).fetchone()[0]
    corr_results.append((order_id, total))
ms_corr = (time.perf_counter() - start) * 1000
print(f"  correlated: {len(corr_results)} rows in {ms_corr:.1f}ms (ran subquery {len(order_sample)}× — O(N²))")

print()
print("=== JOIN Rewrite (materialized aggregation) ===")
start = time.perf_counter()
# step 1: compute aggregation ONCE — this is what CTE/GROUP BY does
cust_totals = dict(cur.execute(
    "SELECT customer_id, SUM(amount) FROM orders GROUP BY customer_id"
).fetchall())
# step 2: single pass through outer rows, O(1) dict lookup
join_results = [(oid, cust_totals.get(cid, 0.0)) for oid, cid, _ in order_sample]
ms_join = (time.perf_counter() - start) * 1000
print(f"  join rewrite: {len(join_results)} rows in {ms_join:.1f}ms (aggregation ran 1×)")
print(f"  speedup: {ms_corr/max(ms_join,0.001):.1f}×")

print()
print("=== CTE reuse — avoid duplicate subquery execution ===")
# demonstrate with actual SQL using sqlite3
start = time.perf_counter()
cte_result = cur.execute("""
    WITH high_value AS (
        SELECT customer_id, SUM(amount) AS total_amount
        FROM orders
        WHERE status = 'CLOSED'
        GROUP BY customer_id
        HAVING SUM(amount) > 5000
    )
    SELECT c.name, hv.total_amount
    FROM high_value hv
    JOIN customers c ON hv.customer_id = c.customer_id
    ORDER BY hv.total_amount DESC
    LIMIT 5
""").fetchall()
ms_cte = (time.perf_counter() - start) * 1000
print(f"  CTE high_value: top 5 customers by closed-order volume")
for name, total in cte_result:
    print(f"    {name:15s}: ${total:,.2f}")
print(f"  executed in {ms_cte:.2f}ms")

print()
print("Rules:")
print("  1. Correlated subquery in SELECT → rewrite as JOIN + CTE")
print("  2. Same expensive subquery repeated → CTE (evaluated once)")
print("  3. CTE referenced 5+ times → consider temp table (explicit materialization)")

print("\nQuery rewriting pattern complete.")

<a id='10'></a>
## 10. The Query Optimization Decision Map

---

```
SYMPTOM                         ROOT CAUSE              FIX
────────────────────────────────────────────────────────────────────────────
Seq Scan on 1M+ row table       no index on filter col  CREATE INDEX ON t(col)
Index Scan but slow             fetching heap per row   add covering INCLUDE()
Wrong rows estimate             stale stats             ANALYZE table
Nested Loop on large tables     optimizer misestimates  SET enable_nestloop=off
Hash join spilling to disk      work_mem too small      SET work_mem='256MB'
Correlated subquery in SELECT   N² execution            rewrite as JOIN + CTE
Same subquery repeated          not materialized        use CTE or temp table
OFFSET 50000 slow               skip N rows still scans keyset: WHERE id > last_id
LIKE '%word%' not using index   leading wildcard        full-text index (GIN)
Function in WHERE (YEAR(col))   function breaks index   rewrite as range predicate
ORDER BY + LIMIT slow           no matching index       CREATE INDEX ON t(sort_col)
COUNT(*) slow                   no stats                ANALYZE, or approx count
────────────────────────────────────────────────────────────────────────────

EXECUTION PLAN NODE GLOSSARY:
  Seq Scan         → full table scan (O(N))
  Index Scan       → B-tree traversal + heap fetch (O(log N + k))
  Index Only Scan  → B-tree only, no heap (fastest!)
  Bitmap Index Scan→ batch random reads (good for medium selectivity)
  Hash Join        → build + probe (O(N+M))
  Merge Join       → sorted merge (O(N+M) if pre-sorted)
  Nested Loop      → for each outer, scan inner (O(N×M) without index)
  Sort             → may spill to disk if > work_mem
  Materialize      → CTE or subquery cached in memory
  Gather           → parallel workers merge results
```


<a id='11'></a>
## 11. Interview Cheat Sheet

---

### When to reach for Query Optimization:

| Signal | What to Do |
|--------|------------|
| Query > 1s on < 1M rows | EXPLAIN ANALYZE first |
| Filter on non-PK column | Check if index exists |
| JOIN two large tables | Verify join keys are indexed |
| Repeated expensive subquery | Rewrite as CTE |
| SELECT * everywhere | Project only needed columns |
| Pagination with OFFSET | Switch to keyset pagination |

---

### The fast operations — memorize these:

```sql
-- Check plan
EXPLAIN ANALYZE SELECT ...;

-- Create optimal composite index (equality first, range last)
CREATE INDEX idx_name ON table(eq_col1, eq_col2, range_col);

-- Covering index (avoids heap fetch)
CREATE INDEX idx_cov ON table(filter_col) INCLUDE (projected_col1, projected_col2);

-- Partial index (hot subset only)
CREATE INDEX idx_partial ON table(col) WHERE status = 'ACTIVE';

-- Refresh stats
ANALYZE table_name;

-- Keyset pagination (vs slow OFFSET)
SELECT * FROM orders WHERE order_id > :last_id ORDER BY order_id LIMIT 100;
```

---

### Common templates:

```sql
-- TEMPLATE: Correlated subquery → JOIN rewrite
-- BAD:  SELECT *, (SELECT SUM(x) FROM t2 WHERE t2.id=t1.id) FROM t1
-- GOOD:
WITH agg AS (SELECT id, SUM(x) AS s FROM t2 GROUP BY id)
SELECT t1.*, agg.s FROM t1 JOIN agg ON t1.id = agg.id;

-- TEMPLATE: Window function avoids self-join
SELECT order_id, SUM(amount) OVER (PARTITION BY customer_id) AS cust_total
FROM orders;

-- TEMPLATE: Avoid function on indexed column
-- BAD:  WHERE YEAR(order_date) = 2024
-- GOOD:
WHERE order_date BETWEEN '2024-01-01' AND '2024-12-31';
```

---

### Gotchas to not forget:

```
❌  Index on low-cardinality column (4 statuses) — full scan beats index
❌  Function in WHERE clause — always rewrites predicate to avoid function on column
❌  Implicit type cast in JOIN (INT vs VARCHAR) — kills index use
❌  UPDATE/DELETE without WHERE — full table lock + seq scan
✅  ANALYZE after large data load — stats control all optimizer decisions
✅  Covering index = zero heap fetches = fastest possible read
✅  CTE for correlated subquery = O(N) instead of O(N²)
✅  EXPLAIN shows estimates; EXPLAIN ANALYZE shows actuals — always use ANALYZE
```


<a id='12'></a>
## 12. Summary Map

---

```
                    🔍 QUERY OPTIMIZATION
                             │
           ┌─────────────────┼─────────────────────┐
           │                 │                     │
     EXECUTION          INDEX                 QUERY
     PLANS              SELECTION             REWRITING
     (Pattern 1)        (Pattern 2)           (Pattern 5)
           │                 │                     │
    EXPLAIN ANALYZE    Single col          CTE (materialize once)
    Seq vs Index Scan  Composite           Correlated → JOIN
    rows estimate      (eq first, range    Temp table
    vs actual          last)
                       Covering (INCLUDE)
                       Partial (WHERE)
           │                 │
     PARTITION          JOIN
     PRUNING            STRATEGIES
     (Pattern 3)        (Pattern 4)
           │                 │
    Prune partitions   Nested Loop  O(N×M)
    before scan        Hash Join    O(N+M)
    Predicate          Merge Join   O(N+M) sorted
    pushdown           → build smaller table

OPTIMIZATION PRIORITY ORDER:
  1. EXPLAIN ANALYZE — find the bottleneck node
  2. Check stats — ANALYZE if estimates are wrong
  3. Add index — if Seq Scan on selective filter
  4. Rewrite query — correlated subquery → CTE/JOIN
  5. Tune join — force hash join if nested loop chosen badly
  6. Partition — partition pruning for time-series tables
```

---
*End of Query Optimization Master Guide — Sean Edition*
